In [27]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [28]:
# Relevant health & wellness documents
docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source":"I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [29]:
# Initial Embeddings
embedding_model = OpenAIEmbeddings()

# Create Vector Store
vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [30]:
# Create Retrievers - Just for comparision
similarity_retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k':5}
)

In [43]:
# Create Multi-Query Retriever
# Here, unlike other retrievers, MultiQueryRetriever is used to create mqr
# Use from_llm() and pass the required parameters
#   retriever: execute the vector_store.as_retriever() function with k value
#     This retriever defined here, will be used to sematic search each query for k queries. 
#     It can any retriever like similarity search or mmr or aany other
#   k value here, is the number of query which will be generated from LLM
#   llm: Provide the Chat model ot generate k similar queries for better query understanding

mqr = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={'k':4}),
    llm=ChatOpenAI()
)

In [44]:
# Providing query
query = 'How to improve enery levels and maintain balance'

In [45]:
# Retrieving using ordinary retriever (here, similarity search)
result = similarity_retriever.invoke(query)
[print(doc.page_content) for doc in result]

# Answer related to Solar system is not correct in this context

Drinking sufficient water throughout the day helps maintain metabolism and energy.
The solar energy system in modern homes helps balance electricity demand.
Consuming leafy greens and fruits helps detox the body and improve longevity.
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Regular walking boosts heart health and can reduce symptoms of depression.


[None, None, None, None, None]

In [46]:
# Retrieving using MQR 
result = mqr.invoke(query)
[print(doc.page_content) for doc in result]

# Here the top 5 outputs are more relevant

Drinking sufficient water throughout the day helps maintain metabolism and energy.
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Regular walking boosts heart health and can reduce symptoms of depression.
Deep sleep is crucial for cellular repair and emotional regulation.
Consuming leafy greens and fruits helps detox the body and improve longevity.


[None, None, None, None, None]